In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

import gitsummary
print(gitsummary.print_cwd_git_version())
import socket; print(f"Running on: {socket.getfqdn()}")

# ERA5 - TECA BARD v1.0.1

This section uses TECA BARD to determine unambiguous AR presence over Bloomington, IN.

In [ ]:
""" Open the TECA ARTMIP files """

file_glob = f"/global/cfs/cdirs/m4374/user_work_directories/taobrien/teca_bard_era5_redo/teca_bard_era5/ERA5_BARD_AR.*.nc4"

# open the dataset
teca_xr = xr.open_mfdataset(file_glob)

# select times that overlap with MERRA2; also subset to 3-hourly
teca_xr = teca_xr.sel(time = slice('1980-01-01', '2017-06-30')).isel(time = slice(0, None, 3))
teca_xr

In [ ]:
""" Pull out a point over Bloomington """
blat, blon = 39.1653, -86.5264 + 360

teca_bloomington = teca_xr.sel(latitude = blat, longitude = blon, method = 'nearest')
teca_bloomington['indices'] = xr.DataArray(np.arange(0,len(teca_bloomington.time), dtype = int), dims = ('time',))
teca_bloomington.load()


## Comparison with MERRA2 AR times

In [ ]:
""" Open the MERRA2 AR detection file. """
merra2_inds = np.loadtxt("./merra2_index_files/bloomington_ars_annual_teca_bard_v1.0.1.txt", dtype = int)

#
tecaera5_at_merra2_times = teca_bloomington.isel(time = merra2_inds)


In [ ]:
e5arprob_at_merra2_times = tecaera5_at_merra2_times['ar_probability']

e5arprob_at_merra2_times.plot()

In [ ]:
""" Plot a histogram of the AR probability """
import matplotlib.pyplot as plt
plt.hist(e5arprob_at_merra2_times, bins = 30, density = True)
plt.show()

In [ ]:
""" Find times with unambiguous AR conditions """

bloomington_ar = teca_bloomington.where(teca_bloomington['ar_probability'] == 1, drop = True)
bloomington_ar

In [ ]:
""" Find only times in specific seasons """

bloomington_ar_season = {}
for season in ['DJF','MAM','JJA','SON']:
    bloomington_ar_season[season] = bloomington_ar.where(bloomington_ar['time.season'] == season, drop = True)
    # load the dataset
    bloomington_ar_season[season].load()
bloomington_ar_season['annual'] = bloomington_ar    

In [ ]:
""" Write the indices to disk """
algorithm = "teca_bard_v1.0.1"
output_dir = "era5_sensitivity_study"
import os
os.makedirs(output_dir, exist_ok = True)
for season in bloomington_ar_season:
    indices = bloomington_ar_season[season]['indices'].values
    np.savetxt(f"{output_dir}/era5_like_merra2_bloomington_ars_{season}_{algorithm}.txt",indices,fmt = "%i")

# ERA5

This section converts the MERRA2 indices to indices in ERA5, assuming that the dataset is opened from 1979-01-01 00Z at hourly resolution, with time indices increasing linearly.

In [ ]:
""" Create a time axis representing ERA5 """
e5_times = pd.DataFrame(dict(time = pd.date_range('1979-01-01 00:00:00', '2021-12-01 00:00:00', freq = '1H')))
e5_times = e5_times.reindex(e5_times['time'])#.to_xarray()
e5_times['time'] = e5_times.index
e5_times = e5_times.to_xarray()
e5_times['indices'] = xr.DataArray(np.arange(0,len(e5_times.time), dtype = int), dims = ('time',))
e5_times

In [ ]:
""" Convert to ERA5 indices and save. """
for season in bloomington_ar_season:
    e5_indices = e5_times.sel(time = bloomington_ar_season[season].time)['indices'].values
    np.savetxt(f"{output_dir}/bloomington_ars_{season}_era5_{algorithm}.txt",e5_indices,fmt = "%i")